# Credit Score Classification
**Comparing MLP, LSTM, GRU, and Random Forest for automated credit scoring**

Dataset: [Kaggle Credit Score Classification](https://www.kaggle.com/datasets/parisrohan/credit-score-classification)

Authors: Omnia Dafalla, Haley Burger, Zayne Maughan — Utah State University, 2025

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.sequence import pad_sequences
import keras_tuner as kt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('Libraries loaded successfully.')

## 2. Load Data

In [ ]:
# Download from: https://www.kaggle.com/datasets/parisrohan/credit-score-classification
# Place train.csv in the data/ folder
df = pd.read_csv('data/train.csv', low_memory=False)
print(f'Shape: {df.shape}')
print(f'Customers: {df["Customer_ID"].nunique()}')
df.head(3)

## 3. Exploratory Data Analysis

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Target distribution
order = ['Poor', 'Standard', 'Good']
colors = ['#E8A838', '#E8622A', '#D84B6E']
df['Credit_Score'].value_counts()[order].plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Class Distribution of Credit Scores', fontsize=13)
axes[0].set_xlabel('Credit Score Category')
axes[0].set_ylabel('Number of Records')
axes[0].tick_params(rotation=0)

# Outlier boxplots
outlier_cols = ['Total_EMI_per_month', 'Num_Credit_Card', 'Interest_Rate']
df_numeric = df[outlier_cols].apply(pd.to_numeric, errors='coerce')
df_numeric.boxplot(ax=axes[1])
axes[1].set_title('Outlier Detection — Key Features', fontsize=13)
plt.tight_layout()
plt.savefig('results/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Data Cleaning & Feature Engineering

In [ ]:
df_clean = df.copy()

# --- Drop PII columns ---
drop_cols = ['Name', 'SSN', 'ID']
df_clean.drop(columns=[c for c in drop_cols if c in df_clean.columns], inplace=True)

# --- Clean numeric columns ---
numeric_cols = ['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts',
                'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date',
                'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries',
                'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Total_EMI_per_month',
                'Amount_invested_monthly', 'Monthly_Balance']

for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col].astype(str).str.replace('[^0-9.-]', '', regex=True), errors='coerce')

# --- Handle missing values ---
# Monthly_Inhand_Salary: forward/backward fill by customer
df_clean['Monthly_Inhand_Salary'] = df_clean.groupby('Customer_ID')['Monthly_Inhand_Salary'].transform(
    lambda x: x.fillna(method='ffill').fillna(method='bfill')
)
# Credit_Mix: fill with mode
if 'Credit_Mix' in df_clean.columns:
    df_clean['Credit_Mix'].fillna(df_clean['Credit_Mix'].mode()[0], inplace=True)
# Numeric: fill with median
for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

# --- Outlier capping (Z-score > 3 → cap at 5th/95th percentile) ---
for col in numeric_cols:
    if col in df_clean.columns:
        p05 = df_clean[col].quantile(0.05)
        p95 = df_clean[col].quantile(0.95)
        df_clean[col] = df_clean[col].clip(lower=p05, upper=p95)

# --- Feature engineering ---
df_clean['Debt_Income_Ratio'] = df_clean['Outstanding_Debt'] / (df_clean['Annual_Income'] + 1)

# Payment behavior → spending level + payment value
if 'Payment_Behaviour' in df_clean.columns:
    spending_map = {'Low_spent': 0, 'Medium_spent': 1, 'High_spent': 2}
    payment_map = {'Small_value': 0, 'Medium_value': 1, 'Large_value': 2}
    df_clean['Spending_Level_Num'] = df_clean['Payment_Behaviour'].str.extract(r'(Low_spent|Medium_spent|High_spent)')[0].map(spending_map).fillna(1)
    df_clean['Payment_Value_Num'] = df_clean['Payment_Behaviour'].str.extract(r'(Small_value|Medium_value|Large_value)')[0].map(payment_map).fillna(1)
    df_clean.drop(columns=['Payment_Behaviour'], inplace=True)

# Type_of_Loan → count of loan types
if 'Type_of_Loan' in df_clean.columns:
    df_clean['Num_Loan_Types'] = df_clean['Type_of_Loan'].apply(
        lambda x: len(str(x).split(',')) if pd.notna(x) and x != 'Not Specified' else 0
    )
    df_clean.drop(columns=['Type_of_Loan'], inplace=True)

# Month ordinal encoding
month_map = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
             'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}
if 'Month' in df_clean.columns:
    df_clean['Month_ordinal'] = df_clean['Month'].map(month_map).fillna(0).astype(int)
    df_clean.drop(columns=['Month'], inplace=True)

# One-hot encode Occupation
if 'Occupation' in df_clean.columns:
    df_clean = pd.get_dummies(df_clean, columns=['Occupation'], drop_first=True)

# Credit_Mix encoding
if 'Credit_Mix' in df_clean.columns:
    mix_map = {'Bad': 0, 'Standard': 1, 'Good': 2}
    df_clean['Credit_Mix'] = df_clean['Credit_Mix'].map(mix_map).fillna(1)

# Encode target
target_map = {'Poor': 0, 'Standard': 1, 'Good': 2}
df_clean['Credit_Score_Enc'] = df_clean['Credit_Score'].map(target_map)
df_clean.drop(columns=['Credit_Score'], inplace=True)

print(f'Cleaned shape: {df_clean.shape}')
print(f'Target distribution:\n{df_clean["Credit_Score_Enc"].value_counts()}')

## 5. Prepare Datasets

### 5a. Aggregated (for MLP & Random Forest)

In [ ]:
feature_cols = [c for c in df_clean.columns if c not in ['Customer_ID', 'Credit_Score_Enc']]

# Aggregate by customer (mean across months)
agg_df = df_clean.groupby('Customer_ID')[feature_cols].mean()
target_df = df_clean.groupby('Customer_ID')['Credit_Score_Enc'].agg(lambda x: x.mode()[0])
agg_df['target'] = target_df

X_agg = agg_df.drop(columns=['target']).values
y_agg = agg_df['target'].values

scaler = StandardScaler()
X_agg_scaled = scaler.fit_transform(X_agg)

X_train, X_temp, y_train, y_temp = train_test_split(X_agg_scaled, y_agg, test_size=0.30, stratify=y_agg, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

### 5b. Sequences (for LSTM & GRU)

In [ ]:
# Build per-customer time-series sequences
sequences, seq_targets = [], []
for cust_id, group in df_clean.groupby('Customer_ID'):
    seq = group.sort_values('Month_ordinal')[feature_cols].values if 'Month_ordinal' in feature_cols else group[feature_cols].values
    sequences.append(seq)
    seq_targets.append(group['Credit_Score_Enc'].mode()[0])

MAX_LEN = max(len(s) for s in sequences)
X_seq = pad_sequences(sequences, maxlen=MAX_LEN, dtype='float32', padding='post', value=0.0)
y_seq = np.array(seq_targets)

X_tr_s, X_tmp_s, y_tr_s, y_tmp_s = train_test_split(X_seq, y_seq, test_size=0.30, stratify=y_seq, random_state=SEED)
X_vl_s, X_ts_s, y_vl_s, y_ts_s = train_test_split(X_tmp_s, y_tmp_s, test_size=0.50, stratify=y_tmp_s, random_state=SEED)

# Class weights
cw = compute_class_weight('balanced', classes=np.unique(y_tr_s), y=y_tr_s)
class_weights = dict(enumerate(cw))
print(f'Sequence shape: {X_seq.shape}, Class weights: {class_weights}')

## 6. MLP Model

In [ ]:
def build_mlp(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(3, activation='softmax')
    ], name='MLP')
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

mlp = build_mlp(X_train.shape[1])
mlp.summary()

history_mlp = mlp.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20, batch_size=64, verbose=1
)

# Evaluate
y_pred_mlp = np.argmax(mlp.predict(X_test), axis=1)
print(f'\nMLP Test Accuracy: {accuracy_score(y_test, y_pred_mlp):.4f}')
print(classification_report(y_test, y_pred_mlp, target_names=['Poor','Standard','Good']))

In [ ]:
def plot_history(history, title):
    plt.figure(figsize=(8, 4))
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'{title} — Training vs Validation Accuracy')
    plt.xlabel('Epochs'); plt.ylabel('Accuracy'); plt.legend()
    plt.tight_layout()
    plt.savefig(f'results/{title.lower().replace(" ","_")}_accuracy.png', dpi=150)
    plt.show()

def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Poor','Standard','Good'],
                yticklabels=['Poor','Standard','Good'])
    plt.title(f'{title} Confusion Matrix')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(f'results/{title.lower().replace(" ","_")}_cm.png', dpi=150)
    plt.show()

plot_history(history_mlp, 'MLP')
plot_cm(y_test, y_pred_mlp, 'MLP')

## 7. LSTM Model

In [ ]:
def build_lstm(seq_len, n_features, units=112):
    model = keras.Sequential([
        layers.Input(shape=(seq_len, n_features)),
        layers.Masking(mask_value=0.0),
        layers.LSTM(units, return_sequences=True),
        layers.Dropout(0.3),
        layers.BatchNormalization(),
        layers.LSTM(units),
        layers.Dropout(0.3),
        layers.BatchNormalization(),
        layers.Dense(48, activation='relu'),
        layers.Dropout(0.3),
        layers.BatchNormalization(),
        layers.Dense(3, activation='softmax')
    ], name='LSTM')
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

lstm = build_lstm(X_tr_s.shape[1], X_tr_s.shape[2])
lstm.summary()

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history_lstm = lstm.fit(
    X_tr_s, y_tr_s,
    validation_data=(X_vl_s, y_vl_s),
    epochs=50, batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stop], verbose=1
)

y_pred_lstm = np.argmax(lstm.predict(X_ts_s), axis=1)
print(f'\nLSTM Test Accuracy: {accuracy_score(y_ts_s, y_pred_lstm):.4f}')
print(classification_report(y_ts_s, y_pred_lstm, target_names=['Poor','Standard','Good']))

plot_history(history_lstm, 'LSTM')
plot_cm(y_ts_s, y_pred_lstm, 'LSTM')

## 8. GRU Model

In [ ]:
def build_gru(seq_len, n_features):
    model = keras.Sequential([
        layers.Input(shape=(seq_len, n_features)),
        layers.Masking(mask_value=0.0),
        layers.GRU(64, return_sequences=True),
        layers.Dropout(0.3),
        layers.BatchNormalization(),
        layers.GRU(32, return_sequences=True),
        layers.Dropout(0.3),
        layers.BatchNormalization(),
        layers.GRU(32),
        layers.Dropout(0.3),
        layers.BatchNormalization(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.BatchNormalization(),
        layers.Dense(3, activation='softmax')
    ], name='GRU')
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

gru = build_gru(X_tr_s.shape[1], X_tr_s.shape[2])
gru.summary()

history_gru = gru.fit(
    X_tr_s, y_tr_s,
    validation_data=(X_vl_s, y_vl_s),
    epochs=20, batch_size=32,
    class_weight=class_weights,
    verbose=1
)

y_pred_gru = np.argmax(gru.predict(X_ts_s), axis=1)
print(f'\nGRU Test Accuracy: {accuracy_score(y_ts_s, y_pred_gru):.4f}')
print(classification_report(y_ts_s, y_pred_gru, target_names=['Poor','Standard','Good']))

plot_history(history_gru, 'GRU')
plot_cm(y_ts_s, y_pred_gru, 'GRU')

## 9. Random Forest (Baseline)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=3,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=SEED
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print(f'Random Forest Test Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print(classification_report(y_test, y_pred_rf, target_names=['Poor','Standard','Good']))
plot_cm(y_test, y_pred_rf, 'Random Forest')

# Feature importance
feat_imp = pd.Series(rf.feature_importances_, index=agg_df.drop(columns=['target']).columns)
feat_imp.nlargest(15).sort_values().plot(kind='barh', figsize=(8,6), color='steelblue')
plt.title('Top 15 Feature Importances — Random Forest')
plt.tight_layout()
plt.savefig('results/rf_feature_importance.png', dpi=150)
plt.show()

## 10. Model Comparison Summary

In [ ]:
results = pd.DataFrame({
    'Model': ['MLP', 'LSTM', 'GRU', 'Random Forest'],
    'Test Accuracy': [
        accuracy_score(y_test, y_pred_mlp),
        accuracy_score(y_ts_s, y_pred_lstm),
        accuracy_score(y_ts_s, y_pred_gru),
        accuracy_score(y_test, y_pred_rf)
    ]
}).sort_values('Test Accuracy', ascending=False)

print(results.to_string(index=False))

colors = ['#2ecc71' if m == 'Random Forest' else '#3498db' for m in results['Model']]
results.plot(x='Model', y='Test Accuracy', kind='bar', color=colors, figsize=(8,5), legend=False)
plt.title('Model Comparison — Test Accuracy')
plt.ylabel('Accuracy'); plt.ylim(0.4, 0.85)
plt.axhline(y=0.72, color='green', linestyle='--', alpha=0.5, label='Random Forest baseline')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('results/model_comparison.png', dpi=150)
plt.show()